# clip-grad-norm-pre-step — ex2: reimplement clip_grad_norm_ from scratch and match torch's reference

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `clip-grad-norm-pre-step`. Running the final beacon cell reports progress against the `Optimizer: clip_grad_norm pre-step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: clip_grad_norm pre-step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`clip-grad-norm-pre-step`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "clip-grad-norm-pre-step"
DD_SUBTOPIC = "Optimizer: clip_grad_norm pre-step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Reimplement `clip_grad_norm_` from scratch

Ex1 called `torch.nn.utils.clip_grad_norm_`. The deepening move is to WRITE the same function from scratch over a list of parameters with `.grad` attributes. Knowing the internals lets you debug clipping bugs and write per-group variants.

Algorithm (matches PyTorch's reference):
```python
# 1. Collect grads (skip params with .grad is None).
grads = [p.grad for p in params if p.grad is not None]
if not grads:
    return 0.0
# 2. Compute global L2 norm = sqrt(sum_i ||g_i||^2).
total = t.sqrt(sum(g.detach().pow(2).sum() for g in grads))
# 3. Compute scale; ONLY apply if total > max_norm.
if total > max_norm:
    scale = max_norm / (total + 1e-6)
    for g in grads:
        g.mul_(scale)
return total.item()
```

**Return PRE-clip norm.** Same contract as the library function — the value returned is what the norm WAS, not what it became. This is what you log to monitor training stability.

**Skip `None` grads.** Some params (frozen layers, params with no path to the loss) never have a grad. Including `None` in the sum would crash. Skipping them gives the same answer as PyTorch.

**In-place `g.mul_(scale)`.** Don't allocate; the optimizer reads `.grad` by reference after this returns.

### Exercise 2 — reimplement clip_grad_norm_ from scratch and match torch's reference

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the global-L2-norm clipping algorithm by reimplementing `torch.nn.utils.clip_grad_norm_` from scratch — collect grads, compute the global norm, rescale in-place only if total exceeds `max_norm`, return the pre-clip norm.
> Keywords: clip-grad, manual, global-l2-norm, in-place
> ```

**KCs targeted:** `global-l2-norm-from-grad-list`, `in-place-rescale-when-above-threshold`

Implement `ex2_clip_grad_norm_manual(params, max_norm)`. The from-scratch deepening of ex1's library-call variant.

Algorithm (matches PyTorch's reference):
1. Collect `grads = [p.grad for p in params if p.grad is not None]`.
2. If `grads` is empty: return `0.0` (no params have grads).
3. Compute the GLOBAL L2 norm: `total = sqrt(sum_i (g_i ** 2).sum())` — concat all grads' squared sums, then sqrt.
4. If `total > max_norm`: rescale every grad in-place via `g.mul_(max_norm / (total + 1e-6))`. The `+ 1e-6` matches PyTorch's reference (avoids /0 when total is tiny).
5. If `total <= max_norm`: do NOT touch the grads.
6. Return `total.item()` (pre-clip norm as a Python float).

Inputs:
- `params`: iterable of `nn.Parameter` or any tensors with a `.grad` attribute.
- `max_norm`: `float`.

Output: `float` — pre-clip global norm.

Constraints:
- Do NOT call `torch.nn.utils.clip_grad_norm_` or `torch.nn.utils.clip_grad_value_` — write the math.
- Use `.detach()` when reading grads for the norm computation (don't build a graph through the clip).
- The in-place rescale must mutate the original `.grad` tensors (downstream optimizer reads them by reference).

In [ ]:
def ex2_clip_grad_norm_manual(params, max_norm: float) -> float:
    """Reimplement clip_grad_norm_ over a global L2 norm; return pre-clip norm."""
    raise NotImplementedError()


def _test_ex2():
    import math
    import torch.nn.utils as nn_utils

    # === Case 1: norm > max_norm, all grads rescaled by max_norm/total ===
    p1 = t.nn.Parameter(t.zeros(2))
    p2 = t.nn.Parameter(t.zeros(2))
    p1.grad = t.tensor([3.0, 4.0])     # contributes 25 to sumsq
    p2.grad = t.tensor([0.0, 0.0])     # contributes 0
    # Global norm = sqrt(25) = 5.
    total = ex2_clip_grad_norm_manual([p1, p2], max_norm=1.0)
    assert isinstance(total, float), f'must return Python float, got {type(total).__name__}'
    assert math.isclose(total, 5.0, rel_tol=1e-5), f'pre-clip norm should be 5.0, got {total}'
    # Scale = 1.0 / (5.0 + 1e-6) ≈ 0.2. So p1.grad ≈ [0.6, 0.8].
    assert t.allclose(p1.grad, t.tensor([0.6, 0.8]), atol=1e-3), (
        f'p1.grad should be rescaled to [0.6, 0.8], got {p1.grad}'
    )
    assert t.allclose(p2.grad, t.tensor([0.0, 0.0]), atol=1e-6)

    # === Case 2: norm < max_norm, grads UNCHANGED ===
    p = t.nn.Parameter(t.zeros(2))
    p.grad = t.tensor([0.3, 0.4])      # norm = 0.5
    total = ex2_clip_grad_norm_manual([p], max_norm=1.0)
    assert math.isclose(total, 0.5, rel_tol=1e-5)
    # No clipping: grad unchanged.
    assert t.equal(p.grad, t.tensor([0.3, 0.4])), (
        f'no clipping should occur when norm < max_norm; got {p.grad}'
    )

    # === Case 3: matches PyTorch's clip_grad_norm_ exactly ===
    t.manual_seed(42)
    p1 = t.nn.Parameter(t.randn(3, 4))
    p2 = t.nn.Parameter(t.randn(5))
    p1.grad = t.randn(3, 4) * 10
    p2.grad = t.randn(5) * 10
    # Reference: clone the grads and run torch's clip.
    p1_ref = t.nn.Parameter(p1.detach().clone())
    p2_ref = t.nn.Parameter(p2.detach().clone())
    p1_ref.grad = p1.grad.clone()
    p2_ref.grad = p2.grad.clone()
    ref_norm = nn_utils.clip_grad_norm_([p1_ref, p2_ref], max_norm=1.5).item()
    # Now run our implementation.
    our_norm = ex2_clip_grad_norm_manual([p1, p2], max_norm=1.5)
    assert math.isclose(our_norm, ref_norm, rel_tol=1e-4), (
        f'our pre-clip norm {our_norm} != torch ref {ref_norm}'
    )
    assert t.allclose(p1.grad, p1_ref.grad, atol=1e-4), (
        f'rescaled p1.grad mismatch:\nours={p1.grad}\nref ={p1_ref.grad}'
    )
    assert t.allclose(p2.grad, p2_ref.grad, atol=1e-4)

    # === Case 4: None grads skipped silently ===
    p1 = t.nn.Parameter(t.zeros(2))
    p2 = t.nn.Parameter(t.zeros(2))
    p1.grad = t.tensor([3.0, 4.0])
    p2.grad = None                       # frozen / never participated in loss
    total = ex2_clip_grad_norm_manual([p1, p2], max_norm=1.0)
    assert math.isclose(total, 5.0, rel_tol=1e-5)
    # Only p1.grad was rescaled. p2.grad still None.
    assert p2.grad is None

    # === Case 5: all-None grads → return 0.0 ===
    p1 = t.nn.Parameter(t.zeros(2))
    p2 = t.nn.Parameter(t.zeros(2))
    p1.grad = None
    p2.grad = None
    total = ex2_clip_grad_norm_manual([p1, p2], max_norm=1.0)
    assert total == 0.0, f'all-None grads should give 0.0, got {total}'

    # === Case 6: in-place mutation (same data_ptr before and after) ===
    p = t.nn.Parameter(t.zeros(2))
    p.grad = t.tensor([10.0, 0.0])   # norm 10
    ptr_before = p.grad.data_ptr()
    ex2_clip_grad_norm_manual([p], max_norm=1.0)
    assert p.grad.data_ptr() == ptr_before, (
        'must rescale in-place; allocating new grad tensor breaks optimizer reference'
    )

    # === Case 7: direction preserved (scalar rescale only) ===
    p = t.nn.Parameter(t.zeros(3))
    p.grad = t.tensor([6.0, 8.0, 0.0])   # norm 10
    ex2_clip_grad_norm_manual([p], max_norm=2.0)
    # Direction must be preserved.
    actual_norm = p.grad.norm().item()
    assert math.isclose(actual_norm, 2.0, rel_tol=1e-3), (
        f'post-clip norm should be 2.0, got {actual_norm}'
    )
    # Ratio check: orig 6:8:0 → scaled 1.2:1.6:0.
    assert t.allclose(p.grad, t.tensor([1.2, 1.6, 0.0]), atol=1e-3)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_clip_grad_norm_manual(params, max_norm):
    grads = [p.grad for p in params if p.grad is not None]
    if not grads:
        return 0.0
    total_sq = sum((g.detach() ** 2).sum() for g in grads)
    total = total_sq.sqrt()
    if total.item() > max_norm:
        scale = max_norm / (total + 1e-6)
        for g in grads:
            g.mul_(scale)
    return total.item()
```

**Global norm, not per-tensor norms.** The clip is on the CONCATENATED gradient vector. `sqrt(sum_i ||g_i||^2)` is the L2 norm of the flattened concat — equivalent to `sqrt(sumsq across ALL params)`. Per-tensor clipping (norm each tensor independently to max_norm) is a DIFFERENT algorithm and gives different results.

**`+ 1e-6` in the denominator.** Matches PyTorch's reference implementation. Guards against div-by-zero when total is subnormal (rare in practice, but the library guards it so we do too — this is also what makes our output bit-identical to theirs).

**`.mul_(scale)` not `g *= scale`.** Both mutate in place, but `g.mul_(scale)` is the canonical PyTorch idiom and is what the reference does. Avoids the trap where `g = g * scale` REBINDS g to a new tensor (and the optimizer still holds a reference to the OLD one).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()